# 03 - Feature engineering and leakage audit

Feature sets, engineered features, the empirical leakage screen and SQL/Python feature parity.

> Notebooks display **aggregates only** (no individual loan records) and reuse functions from `src/creditiq/`. Educational project - not for real lending decisions.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from creditiq.config import load_config, setup_logging
from creditiq.cli import processed_path, clean_all_path
from creditiq.reporting import figures as figs

setup_logging("WARNING")
cfg = load_config()
figs.set_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
TARGET = cfg["target"]["name"]

In [2]:
from creditiq.features.build_features import resolve_feature_spec, engineer_features, engineered_column_groups, make_preprocessor
from creditiq.features.leakage_checks import audit_table, univariate_auc_screen, forbidden_columns
loans = pd.read_parquet(processed_path(cfg))
specs = {n: resolve_feature_spec(cfg.raw, n) for n in cfg["features"]["sets"]}
for n, s in specs.items():
    nums, cats = engineered_column_groups(s)
    print(f"{n}: {len(s.input_columns)} raw inputs -> {len(nums)} numeric + {len(cats)} nominal features")
    print("   inputs:", s.input_columns)

applicant: 20 raw inputs -> 23 numeric + 2 nominal features
   inputs: ['loan_amnt', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'term', 'emp_length', 'home_ownership', 'purpose', 'earliest_cr_line', 'issue_d']
origination: 25 raw inputs -> 27 numeric + 4 nominal features
   inputs: ['loan_amnt', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'int_rate', 'installment', 'term', 'emp_length', 'home_ownership', 'purpose', 'sub_grade', 'verification_status', 'initial_list_status', 'earliest_cr_line', 'issue_d']


## Availability audit

In [3]:
audit = audit_table(cfg["feature_catalog"], {n: s.model_input_columns for n, s in specs.items()})
display(audit[["column", "category", "used_in_applicant", "used_in_origination"]])
print("Forbidden in every model:", sorted(forbidden_columns(cfg["feature_catalog"])))

,column,category,used_in_applicant,used_in_origination
0,annual_inc,pre_underwriting,True,True
1,collections_12_mths_ex_med,pre_underwriting,True,True
2,delinq_2yrs,pre_underwriting,True,True
3,dti,pre_underwriting,True,True
4,earliest_cr_line,pre_underwriting,True,True
5,emp_length,pre_underwriting,True,True
6,home_ownership,pre_underwriting,True,True
7,inq_last_6mths,pre_underwriting,True,True
8,mths_since_last_delinq,pre_underwriting,True,True
9,mths_since_last_record,pre_underwriting,True,True


Forbidden in every model: ['addr_state', 'application_type', 'collection_recovery_fee', 'debt_settlement_flag', 'desc', 'emp_title', 'funded_amnt_inv', 'hardship_flag', 'is_adverse', 'issue_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'last_pymnt_amnt', 'last_pymnt_d', 'loan_status', 'next_pymnt_d', 'out_prncp', 'out_prncp_inv', 'policy_code', 'pymnt_plan', 'recoveries', 'settlement_status', 'split', 'title', 'total_pymnt', 'total_pymnt_inv', 'total_rec_int', 'total_rec_late_fee', 'total_rec_prncp', 'url', 'zip_code']


## Empirical leakage screen (single-column AUC, training split)

In [4]:
train = loans[loans["split"].astype(str) == "train"]
screen = univariate_auc_screen(train, TARGET, [c for c in loans.columns if c not in (TARGET, "split")])
screen["category"] = screen["column"].map(lambda c: cfg["feature_catalog"].get(c, {}).get("category"))
display(screen.head(15))

,column,n_non_null,auc,separation,category
0,recoveries,188171,0.966732,0.933464,outcome_derived
1,collection_recovery_fee,188171,0.864804,0.729608,outcome_derived
2,total_rec_prncp,188171,0.159912,0.680177,outcome_derived
3,last_pymnt_amnt,188171,0.234778,0.530444,outcome_derived
4,total_pymnt,188171,0.282296,0.435407,outcome_derived
5,total_pymnt_inv,188171,0.282342,0.435315,outcome_derived
6,last_pymnt_d,188021,0.322302,0.355396,outcome_derived
7,int_rate,188171,0.663386,0.326772,at_origination
8,last_credit_pull_d,188161,0.376952,0.246097,outcome_derived
9,dti,188171,0.563007,0.126013,pre_underwriting


## Engineered features (training split summary)

In [5]:
spec = specs["applicant"]
feats = engineer_features(train, spec)
display(feats.describe(percentiles=[0.01, 0.5, 0.99]).T)

,count,mean,std,min,1%,50%,99%,max
loan_amnt,188171.0,14354.092953,8114.818032,1000.000000,1800.000000,12175.000000,35000.000000,35000.000000
log_annual_inc,188171.0,11.051271,0.506890,8.476580,9.902785,11.034906,12.345839,15.781472
dti,188171.0,17.059932,7.597555,0.000000,1.910000,16.780000,33.660000,34.990000
delinq_2yrs,188171.0,0.239622,0.703733,0.000000,0.000000,0.000000,3.000000,29.000000
inq_last_6mths,188171.0,0.803695,1.032947,0.000000,0.000000,0.000000,4.000000,8.000000
mths_since_last_delinq,80603.0,34.978413,21.612903,0.000000,2.000000,32.000000,81.000000,156.000000
mths_since_last_record,17473.0,86.012076,26.434305,1.000000,11.000000,95.000000,119.000000,121.000000
mths_since_last_major_derog,32514.0,41.779264,20.974438,0.000000,5.000000,41.000000,81.000000,165.000000
open_acc,188171.0,11.000813,4.607518,0.000000,3.000000,10.000000,24.000000,62.000000
pub_rec,188171.0,0.106212,0.406341,0.000000,0.000000,0.000000,1.000000,54.000000


## Train-only preprocessing

In [6]:
pre = make_preprocessor("linear", spec, cfg["model"]["logistic_regression"]).fit(train)
num_pipe = pre.named_steps["encode"].named_transformers_["num"]
nums, _ = engineered_column_groups(spec)
display(pd.DataFrame({"feature": nums, "clip_low": num_pipe.named_steps["clip"].lower_,
                      "clip_high": num_pipe.named_steps["clip"].upper_,
                      "impute_median": num_pipe.named_steps["impute"].statistics_}).head(12))
print("All statistics above were learned from the training split only.")

,feature,clip_low,clip_high,impute_median
0,loan_amnt,1300.000000,35000.000000,12175.000000
1,log_annual_inc,9.680406,12.542548,11.034906
2,dti,1.130000,34.270000,16.780000
3,delinq_2yrs,0.000000,4.000000,0.000000
4,inq_last_6mths,0.000000,5.000000,0.000000
5,mths_since_last_delinq,1.000000,81.000000,32.000000
6,mths_since_last_record,7.000000,119.000000,95.000000
7,mths_since_last_major_derog,3.000000,82.000000,41.000000
8,open_acc,3.000000,26.000000,10.000000
9,pub_rec,0.000000,2.000000,0.000000


All statistics above were learned from the training split only.


## SQL feature queries (DuckDB) agree with Python

In [7]:
from creditiq.analytics.duckdb_queries import connect, parse_sql_file
con = connect({"loans_all": clean_all_path(cfg), "loans": processed_path(cfg)})
q = parse_sql_file(cfg.path("sql_dir") / "03_feature_queries.sql")
sql_feats = con.execute(q["engineered_features"]).df()
py_feats = engineer_features(loans, spec)
for c in sql_feats.columns:
    ok = np.allclose(sql_feats[c].astype(float), py_feats[c].astype(float), equal_nan=True)
    print(f"{c:<24} match={ok}")
display(con.execute(q["dti_decile_risk"]).df())
display(con.execute(q["loan_to_income_quintile_risk"]).df())

loan_to_income           match=True
revol_bal_to_income      match=True
open_to_total_acc        match=True
log_annual_inc           match=True
term_months              match=True
emp_length_years         match=True
credit_history_months    match=True
has_delinq_history       match=True
has_public_record        match=True


,dti_decile,n_loans,min_dti,max_dti,adverse_rate
0,1,18818,0.00,7.13,0.1125
1,2,18817,7.13,10.09,0.1250
2,3,18817,10.09,12.50,0.1332
3,4,18817,12.50,14.64,0.1388
4,5,18817,14.64,16.78,0.1512
5,6,18817,16.78,18.95,0.1649
6,7,18817,18.95,21.29,0.1731
7,8,18817,21.29,23.98,0.1818
8,9,18817,23.98,27.59,0.1912
9,10,18817,27.59,34.99,0.2049


,lti_quintile,n_loans,min_loan_to_income,max_loan_to_income,adverse_rate
0,1,37635,0.002,0.120,0.1072
1,2,37634,0.120,0.180,0.1232
2,3,37634,0.180,0.241,0.1433
3,4,37634,0.241,0.325,0.1809
4,5,37634,0.325,0.713,0.2337
